In [2]:
# Code Generated by Sidekick is for learning and experimentation purposes only.
import sys
import subprocess
import json
import pickle
import textwrap
from pathlib import Path
from datetime import datetime, timezone
from xml.sax.saxutils import escape

try:
    import numpy as np
    import pandas as pd
    from scipy.sparse import csr_matrix
    from sklearn.decomposition import TruncatedSVD
    from sklearn.feature_extraction.text import TfidfVectorizer

    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
    from reportlab.lib.units import inch
    from reportlab.platypus import (
        SimpleDocTemplate,
        Paragraph,
        Spacer,
        Table,
        TableStyle,
        PageBreak,
        Preformatted,
    )
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "numpy", "pandas", "scipy", "scikit-learn", "reportlab", "pyarrow"
    ])
    import numpy as np
    import pandas as pd
    from scipy.sparse import csr_matrix
    from sklearn.decomposition import TruncatedSVD
    from sklearn.feature_extraction.text import TfidfVectorizer

    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
    from reportlab.lib.units import inch
    from reportlab.platypus import (
        SimpleDocTemplate,
        Paragraph,
        Spacer,
        Table,
        TableStyle,
        PageBreak,
        Preformatted,
    )

# ============================================================
# 1) SETTINGS
# ============================================================
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_ROOT = PROJECT_ROOT / "data" / "raw"
BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze"
PREPARED_ROOT = PROJECT_ROOT / "data" / "prepared"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
SILVER_ROOT = PROJECT_ROOT / "data" / "silver"
REPORTS_DIR = PROJECT_ROOT / "reports"
VALIDATION_DIR = REPORTS_DIR / "validation"
LOGS_DIR = PROJECT_ROOT / "logs"
SRC_DIR = PROJECT_ROOT / "src"

MODEL_ROOT = PROJECT_ROOT / "models"
MODEL_ARTIFACTS_DIR = MODEL_ROOT / "artifacts"
MODEL_TRACKING_DIR = MODEL_ROOT / "tracking"
MODEL_REPORTS_DIR = MODEL_ROOT / "reports"

for folder in [MODEL_ROOT, MODEL_ARTIFACTS_DIR, MODEL_TRACKING_DIR, MODEL_REPORTS_DIR, LOGS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

UTC_NOW = datetime.now(timezone.utc)
RUN_ID = UTC_NOW.strftime("%Y%m%dT%H%M%SZ")
RUN_TS = UTC_NOW.isoformat()

OUTPUT_PATH = PROJECT_ROOT / "09 Model Training and Evaluation- DM4ML-Group51.pdf"

MODEL_LOG_FILE = LOGS_DIR / f"model_training_log_{RUN_ID}.jsonl"
MODEL_RUN_JSON = MODEL_TRACKING_DIR / f"model_run_{RUN_ID}.json"
MODEL_RUNS_CSV = MODEL_TRACKING_DIR / "model_runs_registry.csv"
MODEL_METRICS_CSV = MODEL_REPORTS_DIR / f"model_metrics_{RUN_ID}.csv"
MODEL_PER_USER_METRICS_CSV = MODEL_REPORTS_DIR / f"per_user_metrics_{RUN_ID}.csv"
TRAINING_DEMO_CSV = MODEL_REPORTS_DIR / f"sample_recommendations_{RUN_ID}.csv"

COLLAB_MODEL_PKL = MODEL_ARTIFACTS_DIR / f"collaborative_svd_{RUN_ID}.pkl"
CONTENT_MODEL_PKL = MODEL_ARTIFACTS_DIR / f"content_tfidf_{RUN_ID}.pkl"

# PERFORMANCE CONTROLS
MAX_EVENTS_ROWS = 250_000
MAX_PRODUCTS_ROWS = 30_000
MAX_EVAL_USERS = 1500
TOP_K = 10
ENABLE_CONTENT_MODEL = True

# ============================================================
# 2) FAST HELPERS
# ============================================================
def log_event(stage, status, message, extra=None):
    rec = {
        "event_ts": datetime.now(timezone.utc).isoformat(),
        "stage": stage,
        "status": status,
        "message": message,
        "extra": extra or {},
    }
    with open(MODEL_LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def rel_path(path):
    try:
        return str(Path(path).resolve().relative_to(PROJECT_ROOT.resolve()))
    except Exception:
        return str(path)

def latest_file(base_dir, patterns):
    base_dir = Path(base_dir)
    if not base_dir.exists():
        return None
    hits = []
    for pattern in patterns:
        hits.extend(base_dir.rglob(pattern))
    hits = [p for p in hits if p.is_file()]
    return max(hits, key=lambda p: p.stat().st_mtime) if hits else None

def file_info(path):
    if not path or not Path(path).exists():
        return None
    path = Path(path)
    st = path.stat()
    return {
        "name": path.name,
        "relative_path": rel_path(path),
        "modified": datetime.fromtimestamp(st.st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
        "size_kb": round(st.st_size / 1024, 2),
        "suffix": path.suffix.lower(),
    }

def read_text_preview(path, max_lines=50, max_chars=5000):
    if not path or not Path(path).exists():
        return "File not found."
    text = Path(path).read_text(encoding="utf-8", errors="ignore")
    return "\n".join(text.splitlines()[:max_lines])[:max_chars]

def read_json_preview(path, max_chars=5000):
    if not path or not Path(path).exists():
        return "JSON file not found."
    try:
        obj = json.loads(Path(path).read_text(encoding="utf-8", errors="ignore"))
        return json.dumps(obj, indent=2, ensure_ascii=False)[:max_chars]
    except Exception as e:
        return f"Could not read JSON preview: {e}"

def wrap_block_text(text, width=95):
    out = []
    for line in str(text).splitlines():
        if not line.strip():
            out.append("")
            continue
        out.extend(textwrap.wrap(
            line,
            width=width,
            break_long_words=True,
            break_on_hyphens=True,
            replace_whitespace=False,
            drop_whitespace=False
        ) or [""])
    return "\n".join(out)

def wrap_path_for_pdf(value, max_chunk=30):
    if value is None:
        return ""
    text = str(value).strip()
    if not text:
        return ""
    seps = {"\\", "/", "_", "-", "="}
    parts, token = [], ""
    for ch in text:
        token += ch
        if ch in seps:
            parts.append(token)
            token = ""
    if token:
        parts.append(token)

    lines, current = [], ""
    for part in parts:
        if len(current) + len(part) <= max_chunk:
            current += part
        else:
            if current:
                lines.append(current)
            if len(part) <= max_chunk:
                current = part
            else:
                chunks = textwrap.wrap(part, width=max_chunk, break_long_words=True, break_on_hyphens=True)
                if chunks:
                    lines.extend(chunks[:-1])
                    current = chunks[-1]
                else:
                    current = part
    if current:
        lines.append(current)
    return "<br/>".join(escape(x) for x in lines)

def wrap_general_text_for_pdf(value, max_len=36):
    if value is None:
        return ""
    text = str(value).strip()
    if not text:
        return ""
    words = text.split()
    lines, current = [], ""
    for word in words:
        candidate = f"{current} {word}".strip()
        if len(candidate) <= max_len:
            current = candidate
        else:
            if current:
                lines.append(current)
            if len(word) > max_len:
                chunks = textwrap.wrap(word, width=max_len, break_long_words=True, break_on_hyphens=True)
                if chunks:
                    lines.extend(chunks[:-1])
                    current = chunks[-1]
                else:
                    current = word
            else:
                current = word
    if current:
        lines.append(current)
    return "<br/>".join(escape(x) for x in lines)

def make_display_value(v):
    if isinstance(v, np.ndarray):
        return json.dumps(v.tolist(), ensure_ascii=False)
    if isinstance(v, (list, tuple, set)):
        try:
            return json.dumps(list(v), ensure_ascii=False)
        except Exception:
            return str(v)
    if isinstance(v, dict):
        try:
            return json.dumps(v, ensure_ascii=False, sort_keys=True, default=str)
        except Exception:
            return str(v)
    try:
        if pd.isna(v):
            return ""
    except Exception:
        pass
    return str(v)

# ============================================================
# 3) TARGETED DISCOVERY ONLY
# ============================================================
def discover_events_file():
    search_roots = [PREPARED_ROOT, PROCESSED_ROOT, SILVER_ROOT, RAW_ROOT]
    patterns = [
        "prepared_interactions*.parquet",
        "prepared_interactions*.csv",
        "*interaction*.csv",
        "*events*.csv",
        "events.csv",
    ]
    hits = []
    for root in search_roots:
        if root.exists():
            for p in patterns:
                hits.extend(root.rglob(p))
    hits = [x for x in hits if x.is_file()]
    return max(hits, key=lambda p: p.stat().st_mtime) if hits else None

def discover_products_file():
    search_roots = [BRONZE_ROOT, RAW_ROOT]
    patterns = [
        "products.parquet",
        "products.csv",
        "products_raw.json",
        "*product*.parquet",
        "*product*.csv",
        "*product*.json",
    ]
    hits = []
    for root in search_roots:
        if root.exists():
            for p in patterns:
                hits.extend(root.rglob(p))
    hits = [x for x in hits if x.is_file()]
    return max(hits, key=lambda p: p.stat().st_mtime) if hits else None

def find_model_assets_fast(limit=5):
    # only src, not full repo
    if not SRC_DIR.exists():
        return []
    patterns = ["*train*.py", "*evaluate*.py", "*model*.py", "*recomm*.py", "*svd*.py"]
    hits = []
    for p in patterns:
        hits.extend(SRC_DIR.rglob(p))
    hits = [x for x in sorted(set(hits)) if x.is_file()]
    return hits[:limit]

# ============================================================
# 4) FAST LOADERS
# ============================================================
def load_events_fast(path, max_rows=MAX_EVENTS_ROWS):
    if not path or not Path(path).exists():
        return None

    path = Path(path)
    usecols = None

    if path.suffix.lower() == ".csv":
        header = pd.read_csv(path, nrows=0)
        cols = set(header.columns)
        wanted = []
        for c in header.columns:
            cl = c.lower()
            if cl in {"visitorid", "userid", "user_id", "itemid", "productid", "item_id", "timestamp", "event_ts", "event", "eventtype", "rating", "score"}:
                wanted.append(c)
        usecols = wanted if wanted else None
        return pd.read_csv(path, usecols=usecols, nrows=max_rows)

    if path.suffix.lower() == ".parquet":
        sample = pd.read_parquet(path, columns=None).head(0)
        wanted = []
        for c in sample.columns:
            cl = c.lower()
            if cl in {"visitorid", "userid", "user_id", "itemid", "productid", "item_id", "timestamp", "event_ts", "event", "eventtype", "rating", "score"}:
                wanted.append(c)
        df = pd.read_parquet(path, columns=wanted if wanted else None)
        return df.head(max_rows)

    return None

def load_products_fast(path, max_rows=MAX_PRODUCTS_ROWS):
    if not path or not Path(path).exists():
        return None

    path = Path(path)

    if path.suffix.lower() == ".csv":
        header = pd.read_csv(path, nrows=0)
        wanted = []
        for c in header.columns:
            cl = c.lower()
            if cl in {"id", "product_id", "title", "category", "description", "brand", "price"}:
                wanted.append(c)
        return pd.read_csv(path, usecols=wanted if wanted else None, nrows=max_rows)

    if path.suffix.lower() == ".parquet":
        sample = pd.read_parquet(path, columns=None).head(0)
        wanted = []
        for c in sample.columns:
            cl = c.lower()
            if cl in {"id", "product_id", "title", "category", "description", "brand", "price"}:
                wanted.append(c)
        df = pd.read_parquet(path, columns=wanted if wanted else None)
        return df.head(max_rows)

    if path.suffix.lower() == ".json":
        obj = json.loads(path.read_text(encoding="utf-8", errors="ignore"))
        if isinstance(obj, dict) and "products" in obj and isinstance(obj["products"], list):
            df = pd.DataFrame(obj["products"])
        elif isinstance(obj, list):
            df = pd.DataFrame(obj)
        else:
            df = pd.DataFrame([obj])

        keep = [c for c in df.columns if c.lower() in {"id", "product_id", "title", "category", "description", "brand", "price"}]
        return df[keep].head(max_rows) if keep else df.head(max_rows)

    return None

# ============================================================
# 5) PREP
# ============================================================
def standardize_events(df):
    if df is None or df.empty:
        return None, []

    notes = []
    df = df.copy()
    lower_map = {c.lower(): c for c in df.columns}
    rename = {}

    for src, dst in [
        ("visitorid", "user_id"),
        ("userid", "user_id"),
        ("itemid", "item_id"),
        ("productid", "item_id"),
        ("timestamp", "event_ts"),
        ("eventtype", "event_type"),
        ("event", "event_type"),
        ("score", "rating"),
    ]:
        if src in lower_map and dst not in df.columns:
            rename[lower_map[src]] = dst

    df = df.rename(columns=rename)
    notes.append("Standardized interaction columns.")

    needed = [c for c in ["user_id", "item_id"] if c in df.columns]
    if len(needed) < 2:
        return None, ["user_id/item_id not available in interaction file."]

    df = df.dropna(subset=needed)
    df["user_id"] = df["user_id"].astype(str)
    df["item_id"] = df["item_id"].astype(str)

    if "event_type" not in df.columns:
        df["event_type"] = "interaction"
    else:
        df["event_type"] = df["event_type"].astype(str).str.strip().str.lower()

    if "event_ts" in df.columns:
        ts_num = pd.to_numeric(df["event_ts"], errors="coerce")
        if ts_num.notna().any():
            median_val = ts_num.dropna().median()
            if median_val > 1e12:
                df["event_ts"] = pd.to_datetime(ts_num, unit="ms", errors="coerce")
            elif median_val > 1e9:
                df["event_ts"] = pd.to_datetime(ts_num, unit="s", errors="coerce")
            else:
                df["event_ts"] = pd.to_datetime(df["event_ts"], errors="coerce")
        else:
            df["event_ts"] = pd.to_datetime(df["event_ts"], errors="coerce")
    else:
        df["event_ts"] = pd.NaT

    if "rating" in df.columns:
        df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

    weight_map = {
        "view": 1.0,
        "click": 1.0,
        "interaction": 1.0,
        "addtocart": 3.0,
        "cart": 3.0,
        "purchase": 5.0,
        "transaction": 5.0,
    }
    df["feedback"] = df["event_type"].map(weight_map).fillna(1.0)
    if "rating" in df.columns and df["rating"].notna().any():
        df["feedback"] = np.where(df["rating"].notna(), df["rating"], df["feedback"])

    dedupe_cols = [c for c in ["user_id", "item_id", "event_type", "event_ts"] if c in df.columns]
    df = df.drop_duplicates(subset=dedupe_cols)

    # hard cap again after cleanup
    if len(df) > MAX_EVENTS_ROWS:
        df = df.sort_values(["event_ts"] if "event_ts" in df.columns else ["user_id"]).tail(MAX_EVENTS_ROWS)

    notes.append(f"Prepared {len(df)} interaction rows.")
    return df, notes

def standardize_products(df):
    if df is None or df.empty:
        return None, []

    notes = []
    df = df.copy()
    lower_map = {c.lower(): c for c in df.columns}
    rename = {}

    for src, dst in [
        ("id", "product_id"),
        ("title", "title"),
        ("category", "category"),
        ("description", "description"),
        ("brand", "brand"),
        ("price", "price"),
    ]:
        if src in lower_map and dst not in df.columns:
            rename[lower_map[src]] = dst

    df = df.rename(columns=rename)

    if "product_id" not in df.columns:
        return None, ["product_id not available in products file."]

    df["product_id"] = df["product_id"].astype(str)
    df = df.drop_duplicates(subset=["product_id"])

    for c in ["title", "category", "description", "brand"]:
        if c in df.columns:
            df[c] = df[c].astype(str).fillna("").str.strip()

    if "price" in df.columns:
        df["price"] = pd.to_numeric(df["price"], errors="coerce")

    notes.append(f"Prepared {len(df)} product rows.")
    return df, notes

def leave_one_out_split(df):
    if df is None or df.empty:
        return pd.DataFrame(), pd.DataFrame(), []

    work = df.copy()
    work["_row_id"] = np.arange(len(work))
    counts = work.groupby("user_id").size()
    valid_users = counts[counts >= 2].index
    work = work[work["user_id"].isin(valid_users)].copy()

    if work.empty:
        return pd.DataFrame(), pd.DataFrame(), ["No users with at least 2 interactions."]

    sort_cols = ["user_id", "_row_id"]
    if "event_ts" in work.columns:
        sort_cols = ["user_id", "event_ts", "_row_id"]

    ordered = work.sort_values(sort_cols)
    test_idx = ordered.groupby("user_id").tail(1).index
    test_df = work.loc[test_idx].copy()
    train_df = work.drop(index=test_idx).copy()

    train_df = train_df.drop(columns=["_row_id"])
    test_df = test_df.drop(columns=["_row_id"])

    notes = [f"Train rows: {len(train_df)} | Test rows: {len(test_df)}"]
    return train_df, test_df, notes

# ============================================================
# 6) MODELS
# ============================================================
def build_interaction_matrix(train_df):
    agg = train_df.groupby(["user_id", "item_id"], as_index=False)["feedback"].sum()
    users = agg["user_id"].drop_duplicates().tolist()
    items = agg["item_id"].drop_duplicates().tolist()
    user_to_idx = {u: i for i, u in enumerate(users)}
    item_to_idx = {it: i for i, it in enumerate(items)}

    rows = agg["user_id"].map(user_to_idx).to_numpy()
    cols = agg["item_id"].map(item_to_idx).to_numpy()
    vals = agg["feedback"].astype(float).to_numpy()

    matrix = csr_matrix((vals, (rows, cols)), shape=(len(users), len(items)))
    seen = train_df.groupby("user_id")["item_id"].apply(lambda s: set(s.astype(str))).to_dict()
    popularity = train_df.groupby("item_id")["feedback"].sum().sort_values(ascending=False).index.tolist()

    return matrix, users, items, user_to_idx, item_to_idx, seen, popularity

def train_collaborative_svd(train_df):
    result = {"model_name": "collaborative_svd", "status": "SKIPPED", "notes": [], "params": {}, "artifact_path": None}
    if train_df.empty:
        result["notes"].append("No training data.")
        return result

    matrix, users, items, user_to_idx, item_to_idx, seen, popularity = build_interaction_matrix(train_df)
    n_users, n_items = matrix.shape
    if n_users < 2 or n_items < 2:
        result["notes"].append("Not enough users/items for SVD.")
        return result

    n_components = min(16, max(2, min(n_users, n_items) - 1))
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    user_factors = svd.fit_transform(matrix)

    payload = {
        "svd": svd,
        "user_factors": user_factors,
        "users": users,
        "items": items,
        "user_to_idx": user_to_idx,
        "item_to_idx": item_to_idx,
        "seen": seen,
        "popularity": popularity,
    }

    with open(COLLAB_MODEL_PKL, "wb") as f:
        pickle.dump(payload, f)

    result["status"] = "TRAINED"
    result["artifact_path"] = COLLAB_MODEL_PKL
    result["params"] = {"n_components": n_components, "n_users": n_users, "n_items": n_items, "top_k": TOP_K}
    result["notes"].append("Trained collaborative filtering using TruncatedSVD.")
    result["payload"] = payload
    return result

def recommend_collab(payload, user_id, k=TOP_K):
    user_id = str(user_id)
    if user_id not in payload["user_to_idx"]:
        seen = payload["seen"].get(user_id, set())
        return [x for x in payload["popularity"] if x not in seen][:k]

    uidx = payload["user_to_idx"][user_id]
    scores = payload["user_factors"][uidx].dot(payload["svd"].components_)
    scores = np.asarray(scores).ravel()

    seen = payload["seen"].get(user_id, set())
    for item in seen:
        idx = payload["item_to_idx"].get(item)
        if idx is not None:
            scores[idx] = -np.inf

    ranked = np.argsort(scores)[::-1]
    recs = []
    for idx in ranked:
        item = payload["items"][idx]
        if item not in seen:
            recs.append(item)
        if len(recs) >= k:
            break
    return recs

def train_content_model(train_df, products_df):
    result = {"model_name": "content_tfidf", "status": "SKIPPED", "notes": [], "params": {}, "artifact_path": None}

    if not ENABLE_CONTENT_MODEL:
        result["notes"].append("Content model disabled.")
        return result
    if train_df.empty or products_df is None or products_df.empty:
        result["notes"].append("Missing train/products data.")
        return result

    text_cols = [c for c in ["title", "category", "description", "brand"] if c in products_df.columns]
    if not text_cols:
        result["notes"].append("No text columns for content model.")
        return result

    items_in_train = set(train_df["item_id"].astype(str).unique())
    prod = products_df[products_df["product_id"].astype(str).isin(items_in_train)].copy()
    if prod.empty:
        result["notes"].append("No product overlap with train items.")
        return result

    prod["item_text"] = prod[text_cols].fillna("").astype(str).agg(" ".join, axis=1).str.strip()
    prod = prod[prod["item_text"].str.len() > 0].drop_duplicates(subset=["product_id"])

    if len(prod) > MAX_PRODUCTS_ROWS:
        prod = prod.head(MAX_PRODUCTS_ROWS)

    vec = TfidfVectorizer(max_features=3000, ngram_range=(1, 1), stop_words="english")
    item_matrix = vec.fit_transform(prod["item_text"])

    item_ids = prod["product_id"].astype(str).tolist()
    item_to_idx = {x: i for i, x in enumerate(item_ids)}
    seen = train_df.groupby("user_id")["item_id"].apply(lambda s: set(s.astype(str))).to_dict()
    popularity = train_df.groupby("item_id")["feedback"].sum().sort_values(ascending=False).index.tolist()
    user_item = train_df.groupby(["user_id", "item_id"], as_index=False)["feedback"].sum()

    payload = {
        "vectorizer": vec,
        "item_matrix": item_matrix,
        "item_ids": item_ids,
        "item_to_idx": item_to_idx,
        "seen": seen,
        "popularity": popularity,
        "user_item": user_item,
    }

    with open(CONTENT_MODEL_PKL, "wb") as f:
        pickle.dump(payload, f)

    result["status"] = "TRAINED"
    result["artifact_path"] = CONTENT_MODEL_PKL
    result["params"] = {"max_features": 3000, "ngram_range": "1,1", "n_items_indexed": len(item_ids), "top_k": TOP_K}
    result["notes"].append("Trained content-based filtering using TF-IDF.")
    result["payload"] = payload
    return result

def recommend_content(payload, user_id, k=TOP_K):
    user_id = str(user_id)
    sub = payload["user_item"][payload["user_item"]["user_id"].astype(str) == user_id].copy()
    seen = payload["seen"].get(user_id, set())

    if sub.empty:
        return [x for x in payload["popularity"] if x not in seen][:k]

    sub["item_id"] = sub["item_id"].astype(str)
    sub = sub[sub["item_id"].isin(payload["item_to_idx"])]
    if sub.empty:
        return [x for x in payload["popularity"] if x not in seen][:k]

    idxs = sub["item_id"].map(payload["item_to_idx"]).to_numpy()
    weights = sub["feedback"].astype(float).to_numpy().reshape(-1, 1)
    profile = np.asarray(payload["item_matrix"][idxs].multiply(weights).sum(axis=0)).ravel()

    scores = payload["item_matrix"].dot(profile)
    scores = np.asarray(scores).ravel()

    for item in seen:
        idx = payload["item_to_idx"].get(item)
        if idx is not None:
            scores[idx] = -np.inf

    ranked = np.argsort(scores)[::-1]
    recs = []
    for idx in ranked:
        item = payload["item_ids"][idx]
        if item not in seen:
            recs.append(item)
        if len(recs) >= k:
            break
    return recs

# ============================================================
# 7) FAST EVALUATION
# ============================================================
def sample_test_users(test_df, max_users=MAX_EVAL_USERS):
    users = test_df["user_id"].astype(str).drop_duplicates()
    if len(users) <= max_users:
        return users.tolist()
    return users.sample(max_users, random_state=42).tolist()

def evaluate_model(test_df, recommend_fn, model_name, k=TOP_K):
    if test_df.empty:
        return pd.DataFrame(), {}

    eval_users = sample_test_users(test_df, MAX_EVAL_USERS)
    eval_df = test_df[test_df["user_id"].astype(str).isin(eval_users)].copy()

    rows = []
    for _, row in eval_df.iterrows():
        user_id = str(row["user_id"])
        true_item = str(row["item_id"])
        recs = recommend_fn(user_id, k)

        rank = None
        for i, item in enumerate(recs, start=1):
            if str(item) == true_item:
                rank = i
                break

        hit = 1 if rank is not None else 0
        rows.append({
            "model_name": model_name,
            "user_id": user_id,
            "true_item": true_item,
            "recommended_items": ", ".join(map(str, recs)),
            "hit": hit,
            "rank": "" if rank is None else rank,
            "precision_at_k": hit / k,
            "recall_at_k": float(hit),
            "map_at_k": 0.0 if rank is None else 1.0 / rank,
            "ndcg_at_k": 0.0 if rank is None else 1.0 / np.log2(rank + 1),
        })

    per_user = pd.DataFrame(rows)
    metrics = {
        "model_name": model_name,
        "users_evaluated": len(per_user),
        "precision_at_k": round(float(per_user["precision_at_k"].mean()), 6),
        "recall_at_k": round(float(per_user["recall_at_k"].mean()), 6),
        "map_at_k": round(float(per_user["map_at_k"].mean()), 6),
        "ndcg_at_k": round(float(per_user["ndcg_at_k"].mean()), 6),
        "hit_rate_at_k": round(float(per_user["hit"].mean()), 6),
    }
    return per_user, metrics

# ============================================================
# 8) TRACKING
# ============================================================
def update_registry(flat_record):
    if MODEL_RUNS_CSV.exists():
        old = pd.read_csv(MODEL_RUNS_CSV)
        pd.concat([old, pd.DataFrame([flat_record])], ignore_index=True).to_csv(MODEL_RUNS_CSV, index=False)
    else:
        pd.DataFrame([flat_record]).to_csv(MODEL_RUNS_CSV, index=False)

def write_run_metadata(events_file, products_file, source_notes, split_notes, collab_info, content_info, metrics_df):
    run_record = {
        "run_id": RUN_ID,
        "run_ts": RUN_TS,
        "tracking_tool": "custom_local_registry_equivalent_to_mlflow",
        "events_source": rel_path(events_file) if events_file else None,
        "products_source": rel_path(products_file) if products_file else None,
        "source_notes": source_notes,
        "split_notes": split_notes,
        "collaborative_status": collab_info.get("status"),
        "content_status": content_info.get("status"),
        "collaborative_params": collab_info.get("params", {}),
        "content_params": content_info.get("params", {}),
        "collaborative_artifact": rel_path(collab_info["artifact_path"]) if collab_info.get("artifact_path") else None,
        "content_artifact": rel_path(content_info["artifact_path"]) if content_info.get("artifact_path") else None,
        "metrics": metrics_df.to_dict(orient="records") if not metrics_df.empty else [],
    }
    MODEL_RUN_JSON.write_text(json.dumps(run_record, indent=2, ensure_ascii=False), encoding="utf-8")

    flat = {
        "run_id": RUN_ID,
        "run_ts": RUN_TS,
        "tracking_tool": "custom_local_registry_equivalent_to_mlflow",
        "collaborative_status": collab_info.get("status"),
        "content_status": content_info.get("status"),
        "events_source": rel_path(events_file) if events_file else "",
        "products_source": rel_path(products_file) if products_file else "",
        "metrics_file": rel_path(MODEL_METRICS_CSV) if MODEL_METRICS_CSV.exists() else "",
        "run_json": rel_path(MODEL_RUN_JSON),
    }
    update_registry(flat)

# ============================================================
# 9) EXECUTION
# ============================================================
log_event("pipeline", "started", "Fast model training notebook started")

events_file = discover_events_file()
products_file = discover_products_file()
model_assets = find_model_assets_fast()

events_raw = load_events_fast(events_file, MAX_EVENTS_ROWS)
products_raw = load_products_fast(products_file, MAX_PRODUCTS_ROWS)

events_df, event_notes = standardize_events(events_raw)
products_df, product_notes = standardize_products(products_raw)

source_notes = []
source_notes.extend(event_notes if event_notes else [])
source_notes.extend(product_notes if product_notes else [])

train_df, test_df, split_notes = leave_one_out_split(events_df if events_df is not None else pd.DataFrame())

collab_info = train_collaborative_svd(train_df)
content_info = train_content_model(train_df, products_df)

metrics_rows = []
per_user_frames = []

if collab_info.get("status") == "TRAINED":
    per_user_df, metrics = evaluate_model(
        test_df,
        lambda uid, k=TOP_K: recommend_collab(collab_info["payload"], uid, k),
        "collaborative_svd",
        TOP_K
    )
    if not per_user_df.empty:
        per_user_frames.append(per_user_df)
    metrics_rows.append(metrics)

if content_info.get("status") == "TRAINED":
    per_user_df, metrics = evaluate_model(
        test_df,
        lambda uid, k=TOP_K: recommend_content(content_info["payload"], uid, k),
        "content_tfidf",
        TOP_K
    )
    if not per_user_df.empty:
        per_user_frames.append(per_user_df)
    metrics_rows.append(metrics)

metrics_df = pd.DataFrame(metrics_rows)
per_user_metrics_df = pd.concat(per_user_frames, ignore_index=True) if per_user_frames else pd.DataFrame()

if not metrics_df.empty:
    metrics_df.to_csv(MODEL_METRICS_CSV, index=False)
if not per_user_metrics_df.empty:
    per_user_metrics_df.to_csv(MODEL_PER_USER_METRICS_CSV, index=False)

# small demo only
demo_rows = []
sample_users = test_df["user_id"].astype(str).drop_duplicates().head(10).tolist() if not test_df.empty else []
for uid in sample_users:
    heldout = test_df.loc[test_df["user_id"].astype(str) == uid, "item_id"].astype(str).iloc[0]
    demo_rows.append({
        "user_id": uid,
        "heldout_item": heldout,
        "collaborative_top_k": ", ".join(recommend_collab(collab_info["payload"], uid, TOP_K)) if collab_info.get("status") == "TRAINED" else "",
        "content_top_k": ", ".join(recommend_content(content_info["payload"], uid, TOP_K)) if content_info.get("status") == "TRAINED" else "",
    })
demo_df = pd.DataFrame(demo_rows)
if not demo_df.empty:
    demo_df.to_csv(TRAINING_DEMO_CSV, index=False)

write_run_metadata(events_file, products_file, source_notes, split_notes, collab_info, content_info, metrics_df)

log_event("pipeline", "completed", "Fast model training notebook completed", {
    "rows_loaded_events": 0 if events_df is None else len(events_df),
    "rows_loaded_products": 0 if products_df is None else len(products_df),
    "rows_eval": 0 if test_df.empty else min(test_df["user_id"].nunique(), MAX_EVAL_USERS),
})

# ============================================================
# 10) PDF STYLES
# ============================================================
styles = getSampleStyleSheet()

title_style = ParagraphStyle("CustomTitle", parent=styles["Title"], alignment=TA_CENTER, fontSize=16, leading=20, spaceAfter=14)
meta_style = ParagraphStyle("MetaStyle", parent=styles["Normal"], alignment=TA_LEFT, fontSize=10.2, leading=13, spaceAfter=5)
heading_style = ParagraphStyle("HeadingStyle", parent=styles["Heading2"], alignment=TA_LEFT, fontSize=12, leading=15, spaceAfter=8)
body_style = ParagraphStyle("BodyStyle", parent=styles["BodyText"], alignment=TA_JUSTIFY, fontSize=10.0, leading=14, spaceAfter=8)
bullet_style = ParagraphStyle("BulletStyle", parent=styles["BodyText"], alignment=TA_LEFT, fontSize=10.0, leading=14, leftIndent=14, firstLineIndent=-8, spaceAfter=4)
code_style = ParagraphStyle("CodeStyle", parent=styles["Code"], fontName="Courier", fontSize=7.0, leading=8.4)
table_header_style = ParagraphStyle("TableHeaderStyle", parent=styles["BodyText"], fontName="Helvetica-Bold", fontSize=8.1, leading=9.3)
table_cell_style = ParagraphStyle("TableCellStyle", parent=styles["BodyText"], fontName="Helvetica", fontSize=7.0, leading=8.5)

def to_para(value, style, kind="general"):
    return Paragraph(wrap_path_for_pdf(value), style) if kind == "path" else Paragraph(wrap_general_text_for_pdf(value), style)

def make_wrapped_table(data, col_widths=None, path_cols=None):
    path_cols = path_cols or []
    converted = []
    for r, row in enumerate(data):
        row_cells = []
        for c, cell in enumerate(row):
            st = table_header_style if r == 0 else table_cell_style
            if r == 0:
                row_cells.append(Paragraph(escape(str(cell)), st))
            else:
                row_cells.append(to_para(cell, st, "path" if c in path_cols else "general"))
        converted.append(row_cells)

    table = Table(converted, colWidths=col_widths, repeatRows=1)
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#D9EAD3")),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 4),
        ("RIGHTPADDING", (0, 0), (-1, -1), 4),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
    ]))
    return table

def df_to_wrapped_table(df, max_rows=12):
    if df is None or df.empty:
        return make_wrapped_table([["No data available"]], col_widths=[6.8 * inch])
    preview = df.head(max_rows).copy()
    for col in preview.columns:
        preview[col] = preview[col].map(make_display_value)
    data = [list(preview.columns)] + preview.astype(str).values.tolist()
    return make_wrapped_table(data)

# ============================================================
# 11) TARGETED ARTIFACTS FOR PDF ONLY
# ============================================================
latest_validation_txt = latest_file(PROJECT_ROOT, ["**/run_validation.txt"])
latest_validation_json = latest_file(VALIDATION_DIR, ["data_quality_report_*.json"])
latest_validation_log = latest_file(LOGS_DIR, ["validation_log_*.jsonl"])

artifact_rows = [["Artifact", "Relative Path", "Last Modified", "Size"]]
for p in [
    latest_validation_txt,
    latest_validation_json,
    latest_validation_log,
    MODEL_LOG_FILE,
    MODEL_RUN_JSON,
    MODEL_RUNS_CSV,
    MODEL_METRICS_CSV,
    MODEL_PER_USER_METRICS_CSV,
    TRAINING_DEMO_CSV,
    COLLAB_MODEL_PKL if COLLAB_MODEL_PKL.exists() else None,
    CONTENT_MODEL_PKL if CONTENT_MODEL_PKL.exists() else None,
]:
    if p and Path(p).exists():
        info = file_info(p)
        artifact_rows.append([info["name"], info["relative_path"], info["modified"], f"{info['size_kb']} KB"])

asset_rows = [["Training Asset", "Relative Path", "Last Modified", "Size"]]
for p in model_assets:
    info = file_info(p)
    asset_rows.append([info["name"], info["relative_path"], info["modified"], f"{info['size_kb']} KB"])
if len(asset_rows) == 1:
    asset_rows.append(["No src training asset found", "-", "-", "-"])

data_summary_df = pd.DataFrame([
    {"dataset": "events_source", "rows": 0 if events_df is None else len(events_df), "columns": 0 if events_df is None else len(events_df.columns)},
    {"dataset": "train_split", "rows": 0 if train_df is None else len(train_df), "columns": 0 if train_df is None else len(train_df.columns)},
    {"dataset": "test_split", "rows": 0 if test_df is None else len(test_df), "columns": 0 if test_df is None else len(test_df.columns)},
    {"dataset": "products_source", "rows": 0 if products_df is None else len(products_df), "columns": 0 if products_df is None else len(products_df.columns)},
])

run_summary_df = pd.DataFrame([{
    "run_id": RUN_ID,
    "tracking_tool": "custom_local_registry_equivalent_to_mlflow",
    "collaborative_status": collab_info.get("status"),
    "content_status": content_info.get("status"),
    "eval_user_cap": MAX_EVAL_USERS,
    "event_row_cap": MAX_EVENTS_ROWS,
}])

validation_preview = read_text_preview(latest_validation_txt, max_lines=35, max_chars=3500)
validation_json_preview = read_json_preview(latest_validation_json, max_chars=3500)
model_log_preview = read_text_preview(MODEL_LOG_FILE, max_lines=35, max_chars=3500)
run_json_preview = read_json_preview(MODEL_RUN_JSON, max_chars=3500)

# ============================================================
# 12) BUILD PDF
# ============================================================
story = []

story.append(Paragraph("09 Model Training and Evaluation", title_style))
story.append(Paragraph("<b>Course Name:</b> Data Management for Machine Learning", meta_style))
story.append(Paragraph("<b>Assignment Title:</b> End-to-End Data Management Pipeline for a Recommendation System", meta_style))
story.append(Paragraph("<b>Assignment:</b> Group 51 - Data management for Machine Learning Group 51", meta_style))
story.append(Spacer(1, 10))

story.append(Paragraph("<b>Team Members</b>", heading_style))
team_data = [
    ["Team Member Name", "Team Member ID"],
    ["BANSHIDHAR RATH", "2025AE05346"],
    ["JITENDRA KUMAR TIWARI", "2025AE05518"],
    ["KATBA ANKIT CHIMANBHAI", "2025AE05229"],
    ["NAVEEN SURATHU", "2025AE05492"],
]
story.append(make_wrapped_table(team_data, col_widths=[4.0 * inch, 2.0 * inch]))
story.append(Spacer(1, 12))

story.append(Paragraph("1. Objective", heading_style))
story.append(Paragraph(
    "This report captures the fast model training and evaluation workflow for the recommendation system, including model training, offline ranking metrics, and tracked run metadata.",
    body_style
))

story.append(Paragraph("2. Performance Optimizations Applied", heading_style))
for note in [
    "Only targeted folders are scanned instead of the full repository.",
    "Only required columns are loaded from datasets.",
    f"Interaction rows are capped at {MAX_EVENTS_ROWS:,}.",
    f"Product rows are capped at {MAX_PRODUCTS_ROWS:,}.",
    f"Evaluation users are capped at {MAX_EVAL_USERS:,}.",
    "PDF content is limited to small previews and summary tables only.",
]:
    story.append(Paragraph(f"• {escape(note)}", bullet_style))
story.append(Spacer(1, 12))

story.append(Paragraph("3. Artifacts", heading_style))
story.append(make_wrapped_table(
    artifact_rows,
    col_widths=[1.55 * inch, 3.35 * inch, 1.00 * inch, 0.60 * inch],
    path_cols=[1]
))
story.append(Spacer(1, 12))

story.append(Paragraph("4. Training Assets", heading_style))
story.append(make_wrapped_table(
    asset_rows,
    col_widths=[1.65 * inch, 3.25 * inch, 1.00 * inch, 0.60 * inch],
    path_cols=[1]
))
story.append(Spacer(1, 12))

story.append(Paragraph("5. Data Summary", heading_style))
story.append(df_to_wrapped_table(data_summary_df))
story.append(Spacer(1, 12))

story.append(Paragraph("6. Run Summary", heading_style))
story.append(df_to_wrapped_table(run_summary_df))
story.append(Spacer(1, 12))

story.append(PageBreak())

story.append(Paragraph("7. Training Notes", heading_style))
for note in (source_notes + split_notes + collab_info.get("notes", []) + content_info.get("notes", []))[:12]:
    story.append(Paragraph(f"• {escape(note)}", bullet_style))
story.append(Spacer(1, 12))

story.append(Paragraph("8. Model Performance Report", heading_style))
story.append(df_to_wrapped_table(metrics_df, max_rows=10))
story.append(Spacer(1, 12))

story.append(Paragraph("9. Per-User Evaluation Preview", heading_style))
story.append(df_to_wrapped_table(per_user_metrics_df, max_rows=12))
story.append(Spacer(1, 12))

story.append(Paragraph("10. Sample Recommendations", heading_style))
story.append(df_to_wrapped_table(demo_df, max_rows=10))
story.append(Spacer(1, 12))

story.append(PageBreak())

story.append(Paragraph("11. Tracked Run Metadata Preview", heading_style))
story.append(Preformatted(wrap_block_text(run_json_preview, width=95), code_style))
story.append(Spacer(1, 10))

story.append(Paragraph("12. Validation Preview", heading_style))
story.append(Preformatted(wrap_block_text(validation_preview, width=95), code_style))
story.append(Spacer(1, 10))

story.append(Paragraph("13. Data Quality JSON Preview", heading_style))
story.append(Preformatted(wrap_block_text(validation_json_preview, width=95), code_style))
story.append(Spacer(1, 10))

story.append(Paragraph("14. Model Training Log Preview", heading_style))
story.append(Preformatted(wrap_block_text(model_log_preview, width=95), code_style))
story.append(Spacer(1, 10))

story.append(Paragraph("15. Conclusion", heading_style))
story.append(Paragraph(
    "This optimized notebook keeps the required deliverables while removing the main runtime bottlenecks from broad file scanning, oversized previews, full-volume evaluation, and unnecessarily heavy PDF rendering.",
    body_style
))

doc = SimpleDocTemplate(
    str(OUTPUT_PATH),
    pagesize=A4,
    rightMargin=0.50 * inch,
    leftMargin=0.50 * inch,
    topMargin=0.55 * inch,
    bottomMargin=0.55 * inch,
)
doc.build(story)

print(f"PDF created successfully: {OUTPUT_PATH}")
print(f"Run ID: {RUN_ID}")
print(f"Metrics file: {MODEL_METRICS_CSV if MODEL_METRICS_CSV.exists() else 'not created'}")


PDF created successfully: C:\Users\barath\recomart-pipeline\09 Model Training and Evaluation- DM4ML-Group51.pdf
Run ID: 20260430T124222Z
Metrics file: C:\Users\barath\recomart-pipeline\models\reports\model_metrics_20260430T124222Z.csv
